# Import 

In [ ]:
# Mô tả: DViết lại kiến trúc của mô hình Alexnet với thưviện Pytorch. Dựa trên bài học trên lớp, với mỗi mô hình các bạn cần xây dựng 1 class kế thừa 
# từ class nn.Module của Pytorch. Sau đó các bạn cần viết phương thức __init__() và forward() cho class này
# Để đảm bảo mô hình ít nhất không có lỗi kỹ thuật nào, tạo 1 sample data mô phỏng dữ liệu đầu vào (1 tensor 4 chiều - [B,C,H,W]), đưa vào mô hình thực hiện quá trình forward()


import torch
import torch.nn as nn
import torch.nn.functional as F

class AlexNet(nn.Module):
    def __init__(self, num_classes = 1000):
        super().__init__()
        self.features = nn.Sequential(
            # Conv1:
            nn.Conv2d(3, 64, kernel_size = 11, stride = 4, padding = 2),
            nn.ReLU(inplace = True),
            nn.LocalResponseNorm(size = 5, alpha = 1e-4, beta = 0.75, k = 2.0),
            nn.MaxPool2d(kernel_size = 3, stride = 2),

            # Conv2:
            nn.Conv2d(64, 192, kernel_size = 5, padding = 2),
            nn.ReLU(inplace = True),
            nn.LocalResponseNorm(size = 5, alpha = 1e-4, beta = 0.75, k = 2.0),
            nn.MaxPool2d(kernel_size = 3, stride = 2),

            # Conv3–5:
            nn.Conv2d(192, 384, kernel_size = 3, padding = 1),
            nn.ReLU(inplace = True),

            nn.Conv2d(384, 256, kernel_size = 3, padding = 1),
            nn.ReLU(inplace = True),

            nn.Conv2d(256, 256, kernel_size = 3, padding = 1),
            nn.ReLU(inplace = True),

            nn.MaxPool2d(kernel_size = 3, stride = 2),
        )

        # 256 feature maps of 6x6:
        self.classifier = nn.Sequential(
            nn.Dropout(p = 0.5),
            nn.Linear(256 * 6 * 6, 4096),
            nn.ReLU(inplace = True),

            nn.Dropout(p = 0.5),
            nn.Linear(4096, 4096),
            nn.ReLU(inplace = True),

            nn.Linear(4096, num_classes),
        )

        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            # Checks if the layer is a convolution:
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, nonlinearity = "relu")
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0.0)
                    
            # Checks if the layer is a fully connected:
            elif isinstance(m, nn.Linear):
                nn.init.normal_(m.weight, mean = 0.0, std = 0.01)
                nn.init.constant_(m.bias, 0.0)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.features(x)
        x = torch.flatten(x, 1) 
        x = self.classifier(x)
        return x

if __name__ == "__main__":
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = AlexNet(num_classes = 1000).to(device)

    # Example batch: 2 RGB images of size 224x224
    test = torch.randn(2, 3, 224, 224, device = device)

    with torch.no_grad():
        out = model(test)

    # Print shapes:
    print("Input shape : {} x {} x {} x {}".format(test.shape[0], test.shape[1], test.shape[2], test.shape[3]))
    print("Output shape: {} x {}".format(out.shape[0], out.shape[1]))
